In [4]:
import tensorflow as tf
import numpy as np

# Load model
model = tf.keras.models.load_model('../Models/best_model.keras', compile=False)

# Create a random test image
test_img = np.random.rand(1, 100, 100, 1).astype('float32')

# Test 3 times - predictions should be IDENTICAL for the same input
for i in range(3):
    pred = model(test_img, training=False).numpy()
    print(f"Test {i+1}:")
    print(f"  Predicted class: {pred.argmax()}")
    print(f"  Max probability: {pred.max():.6f}")
    print(f"  Top 3 classes: {pred[0].argsort()[-3:][::-1]}")
    print()

Test 1:
  Predicted class: 24
  Max probability: 0.175293
  Top 3 classes: [24 15 16]

Test 2:
  Predicted class: 24
  Max probability: 0.175293
  Top 3 classes: [24 15 16]

Test 3:
  Predicted class: 24
  Max probability: 0.175293
  Top 3 classes: [24 15 16]



In [9]:
import tensorflow as tf
import numpy as np
import os

# Path to your model
MODEL_PATH = '../Models/best_model.keras'  # or 'Models/best_model.keras'

# Path to your data (same as training)
DATA_DIR = '../data'
TRAIN_SUBDIR = 'train'
IMG_SIZE = (100, 100)
BATCH_SIZE = 32
SEED = 1337

print("=" * 50)
print("TESTING MODEL")
print("=" * 50)

# Check if model exists
if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Model not found at {MODEL_PATH}")
    exit()

print(f"\n1. Model file size: {os.path.getsize(MODEL_PATH) / (1024*1024):.2f} MB")

# Load validation data (same as training)
from tensorflow.keras.utils import image_dataset_from_directory

train_dir = os.path.join(DATA_DIR, TRAIN_SUBDIR)

val_ds = image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    color_mode='grayscale',
    validation_split=0.2,
    subset='validation',
    seed=SEED
)

print(f"2. Validation dataset loaded: {len(val_ds)} batches")
print(f"3. Class names: {val_ds.class_names[:10]}...")  # Show first 10

# Load model
print("\n4. Loading model...")
model = tf.keras.models.load_model(MODEL_PATH, compile=False)  # Removed safe_mode
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

# Evaluate on validation data
print("\n5. Evaluating on validation data...")
val_loss, val_acc = model.evaluate(val_ds, verbose=1)

print("\n" + "=" * 50)
print(f"RESULT: Validation Accuracy = {val_acc*100:.2f}%")
print(f"RESULT: Validation Loss = {val_loss:.4f}")
print("=" * 50)

if val_acc > 0.80:
    print("\n✅ Model looks GOOD! Upload this to Hugging Face.")
elif val_acc > 0.50:
    print("\n⚠️  Model is okay but could be better. Consider retraining.")
else:
    print("\n❌ Model is BROKEN! This is the wrong checkpoint. Retrain or use a different checkpoint.")

# Test on a single batch
print("\n6. Testing on a single batch...")
for images, labels in val_ds.take(1):
    predictions = model(images, training=False).numpy()
    pred_classes = predictions.argmax(axis=1)
    true_classes = labels.numpy().argmax(axis=1)
    
    correct = (pred_classes == true_classes).sum()
    print(f"   Batch accuracy: {correct}/{len(pred_classes)} = {correct/len(pred_classes)*100:.1f}%")
    
    # Show first 5 predictions
    print("\n   First 5 predictions:")
    for i in range(min(5, len(pred_classes))):
        true_label = val_ds.class_names[true_classes[i]]
        pred_label = val_ds.class_names[pred_classes[i]]
        confidence = predictions[i][pred_classes[i]] * 100
        status = "✓" if pred_classes[i] == true_classes[i] else "✗"
        print(f"   {status} True: {true_label}, Predicted: {pred_label} ({confidence:.1f}%)")

TESTING MODEL

1. Model file size: 16.43 MB
Found 28800 files belonging to 36 classes.
Using 5760 files for validation.
2. Validation dataset loaded: 180 batches
3. Class names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']...

4. Loading model...
   Input shape: (None, 100, 100, 1)
   Output shape: (None, 36)

5. Evaluating on validation data...
180/180 [==============================] - 3s 12ms/step - loss: 0.4331 - accuracy: 0.9128

RESULT: Validation Accuracy = 91.28%
RESULT: Validation Loss = 0.4331

✅ Model looks GOOD! Upload this to Hugging Face.

6. Testing on a single batch...
   Batch accuracy: 32/32 = 100.0%

   First 5 predictions:
   ✓ True: 7, Predicted: 7 (87.9%)
   ✓ True: v, Predicted: v (99.1%)
   ✓ True: w, Predicted: w (78.0%)
   ✓ True: s, Predicted: s (99.1%)
   ✓ True: 5, Predicted: 5 (97.1%)
